In [ ]:
from pathlib import Path


def find_project_root(start=None):
    current = Path.cwd() if start is None else Path(start).resolve()
    for path in (current, *current.parents):
        if (path / "data").exists():
            return path
    return current


PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / "data"

import gc

import numpy as np
import pandas as pd

INPUT_DIR = DATA_ROOT / "data_train" / "alpha_360F_day"
OUTPUT_DIR = DATA_ROOT / "data_train" / "alpha_360F_day_zscore"
KEYWORD = ""
SUFFIX = "_mad3_zscore_soft"
MODE = "add"
MAD_K = 3.0
USE_FINAL_CLIP = True
FINAL_CLIP_VALUE = 4.0
CODE_PREFIXES = ("0", "3", "6")


def cut_mad(values, mad_k=3.0):
    x = values.astype(float, copy=True)
    x[np.isinf(x)] = np.nan
    median = np.nanmedian(x)
    mad = np.nanmedian(np.abs(x - median))
    if not np.isfinite(median) or not np.isfinite(mad):
        return np.full_like(x, np.nan)
    x[(x > median + mad_k * mad) | (x < median - mad_k * mad)] = np.nan
    return x


def soft_truncate(values, pos_slope, neg_slope):
    return np.where(values > 3, 3 * (1 - pos_slope) + values * pos_slope, np.where(values < -3, -3 * (1 - neg_slope) + values * neg_slope, values))


def slopes_from_z(values):
    vmax = np.nanmax(values)
    vmin = np.nanmin(values)
    pos = np.maximum(0.0, np.minimum(1.0, 0.5 / (vmax - 3.0))) if np.isfinite(vmax) else 0.0
    neg = np.maximum(0.0, np.minimum(1.0, 0.5 / (-vmin - 3.0))) if np.isfinite(vmin) else 0.0
    return float(pos), float(neg)


def fit_params(values):
    x = values.astype(float, copy=True)
    x[np.isinf(x)] = np.nan
    clipped = cut_mad(x, MAD_K)
    mean = np.nanmean(clipped)
    std = np.nanstd(clipped)
    if not np.isfinite(mean) or not np.isfinite(std) or std <= 0:
        return None
    z0 = (x - mean) / std
    pos, neg = slopes_from_z(z0)
    z1 = soft_truncate(z0, pos, neg)
    mean2 = np.nanmean(z1)
    std2 = np.nanstd(z1)
    if not np.isfinite(mean2) or not np.isfinite(std2) or std2 <= 0:
        return None
    return mean, std, pos, neg, mean2, std2


def transform(values, params):
    mean, std, pos, neg, mean2, std2 = params
    x = values.astype(float, copy=True)
    x[np.isinf(x)] = np.nan
    z0 = (x - mean) / std
    z1 = soft_truncate(z0, pos, neg)
    z = (z1 - mean2) / std2
    if USE_FINAL_CLIP and np.isfinite(FINAL_CLIP_VALUE) and FINAL_CLIP_VALUE > 0:
        z = np.clip(z, -FINAL_CLIP_VALUE, FINAL_CLIP_VALUE)
    return z


def code_series(series):
    return series.astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(6)


def filter_codes(frame):
    code_col = "code" if "code" in frame.columns else "Code"
    if code_col not in frame.columns:
        raise ValueError("input parquet must contain a code column")
    codes = code_series(frame[code_col])
    mask = pd.Series(False, index=frame.index)
    for prefix in CODE_PREFIXES:
        mask |= codes.str.startswith(prefix)
    out = frame.loc[mask].copy()
    out[code_col] = codes.loc[mask].values
    return out


def feature_columns(frame):
    excluded = {"date", "Date", "code", "Code"}
    columns = [col for col in frame.columns if col not in excluded]
    if KEYWORD:
        columns = [col for col in columns if KEYWORD in col]
    return [col for col in columns if pd.api.types.is_numeric_dtype(frame[col])]


def process_file(in_path):
    out_path = OUTPUT_DIR / in_path.name
    frame = pd.read_parquet(in_path)
    frame = filter_codes(frame)
    columns = feature_columns(frame)
    mode = MODE.strip().lower()
    if mode not in {"add", "overwrite"}:
        raise ValueError("MODE must be add or overwrite")
    for col in columns:
        values = pd.to_numeric(frame[col], errors="coerce").to_numpy(dtype=float, copy=False)
        params = fit_params(values)
        target_col = f"{col}{SUFFIX}" if mode == "add" else col
        frame[target_col] = np.nan if params is None else transform(values, params)
    if mode == "overwrite":
        frame = frame.rename(columns={col: f"{col}{SUFFIX}" for col in columns})
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    frame.to_parquet(out_path, index=False)
    rows = len(frame)
    del frame
    gc.collect()
    return in_path.name, rows


def main():
    files = sorted(INPUT_DIR.glob("*.parquet"))
    for path in files:
        name, rows = process_file(path)
        print(f"file={name} rows={rows}")
    print(f"done files={len(files)} output={OUTPUT_DIR}")


if __name__ == "__main__":
    main()